## Modeling

### Objective

The goal of this stage is to:

- Train and evaluate machine learning models for churn prediction  
- Establish a baseline model for comparison  
- Handle class imbalance using different strategies  
- Compare multiple models and select the best-performing one  
- Tune hyperparameters to improve model performance  
- Evaluate models using appropriate metrics (precision, recall, F1-score, ROC-AUC)  
- Analyze model performance and interpret key drivers of churn  

## 1. Data Loading & Setup
We begin by importing necessary libraries and loading the dataset.

In [45]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.dummy import DummyClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE

In [47]:
with open('../data/processed/train_test_data.pkl', 'rb') as f:
    X_train, X_test, y_train, y_test = pickle.load(f)

### Evaluation Metrics

In churn prediction, recall is the most critical metric, as it measures the model's ability to correctly identify customers who are likely to leave.

Missing a churned customer (false negative) results in lost revenue, making recall more important than precision.

Precision and F1-score are also considered to balance the trade-off between correctly identifying churn and avoiding unnecessary retention actions.

ROC-AUC is used as an overall measure of model performance.

# 2. Baseline Models

Establish simple baselines for comparison:

- Dummy Classifier (most frequent class)  

These models provide reference performance and help validate the usefulness of more complex models.


In [51]:
# Train
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)

# Predict
y_pred = dummy.predict(X_test)
y_proba = dummy.predict_proba(X_test)[:, 1]

# Metrics
precision = precision_score(y_test, y_pred,zero_division=0)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)
accu = accuracy_score(y_test, y_proba)

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"Accuracy: {accu:.4f}")

Precision: 0.0000
Recall: 0.0000
F1-score: 0.0000
ROC-AUC: 0.5000
Accuracy: 0.7346


The Dummy Classifier achieves an accuracy of 73.46%, which may appear relatively high at first glance. However, this result is misleading due to class imbalance.

The model predicts only the majority class (non-churn), resulting in:

- Recall = 0 → no churned customers are detected  
- Precision = 0 → no positive predictions are made  
- F1-score = 0 → no balance between precision and recall  
- ROC-AUC = 0.5 → performance equivalent to random guessing  

Despite the high accuracy, the model completely fails to identify churned customers, making it ineffective for this task. This highlights the limitation of accuracy as a metric in imbalanced classification problems.

## 3. Evaluation Metrics

Define evaluation metrics:

- Recall (primary metric for churn detection)  
- Precision  
- F1-score  
- ROC-AUC
- Accuracy

Recall is prioritized due to the high cost of missing churned customers.


In [55]:
def get_metrics(y_true, y_pred, y_proba):
    return {
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred),
        "F1-score": f1_score(y_true, y_pred),
        "Accuracy": accuracy_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_proba)
    }


def evaluate_model(model, X_train, y_train, X_test, y_test):
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Probabilities (без проверок)
    y_train_proba = model.predict_proba(X_train)[:, 1]
    y_test_proba = model.predict_proba(X_test)[:, 1]
    
    # Metrics
    train_metrics = get_metrics(y_train, y_train_pred, y_train_proba)
    test_metrics = get_metrics(y_test, y_test_pred, y_test_proba)
    
    # Print
    print("=== Train Metrics ===")
    for k, v in train_metrics.items():
        print(f"{k}: {v:.4f}")
    
    print("\n=== Test Metrics ===")
    for k, v in test_metrics.items():
        print(f"{k}: {v:.4f}")

In [57]:
evaluate_model(dummy, X_train, y_train, X_test, y_test)

=== Train Metrics ===
Precision: 0.0000
Recall: 0.0000
F1-score: 0.0000
Accuracy: 0.7346
ROC-AUC: 0.5000

=== Test Metrics ===
Precision: 0.0000
Recall: 0.0000
F1-score: 0.0000
Accuracy: 0.7346
ROC-AUC: 0.5000


## 4. Handling Class Imbalance

Evaluate different strategies:

- Baseline (no balancing)  
- Class weighting  
- SMOTE (oversampling)  

Compare their impact on model performance.

### 4.1 Baseline

In [61]:
model_baseline = LogisticRegression(max_iter=1000, random_state=42)

model_baseline.fit(X_train, y_train)

print("=== Baseline ===")
evaluate_model(model_baseline, X_train, y_train, X_test, y_test)

=== Baseline ===


ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

### 4.2 Class Weight

In [22]:
model_weighted = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

model_weighted.fit(X_train, y_train)

print("\n=== Class Weight ===")
evaluate_model(model_weighted, X_train, y_train, X_test, y_test)


=== Class Weight ===
=== Train Metrics ===
Precision: 0.5199
Recall: 0.7880
F1-score: 0.6264
Accuracy: 0.7506
ROC-AUC: 0.8431

=== Test Metrics ===
Precision: 0.5080
Recall: 0.7674
F1-score: 0.6113
Accuracy: 0.7410
ROC-AUC: 0.8363


### 4.3 SMOTE

3738    1
3151    1
4860    1
3867    1
3810    0
       ..
6303    2
6227    0
4673    1
2710    1
5639    0
Name: tenure_group, Length: 5634, dtype: int32

In [37]:
X_train.dtypes

gender                                      int64
SeniorCitizen                               int64
Partner                                     int64
Dependents                                  int64
tenure                                    float64
PhoneService                                int64
PaperlessBilling                            int64
avg_monthly_spend                         float64
tenure_group                             category
has_internet                                int64
has_addons                                float64
risky_contract                              int64
auto_payment                                int64
MultipleLines_No phone service              int32
MultipleLines_Yes                           int32
InternetService_Fiber optic                 int32
InternetService_No                          int32
PaymentMethod_Credit card (automatic)       int32
PaymentMethod_Electronic check              int32
PaymentMethod_Mailed check                  int32


In [25]:
smote = SMOTE(random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

In [27]:
model_smote = LogisticRegression(max_iter=1000, random_state=42)

model_smote.fit(X_train_sm, y_train_sm)

print("\n=== SMOTE ===")
evaluate_model(model_smote, X_train_sm, y_train_sm, X_test, y_test)

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## 5. Model Training

Train a diverse set of models:

### Linear Models
- Logistic Regression  

### Tree-Based Models
- Decision Tree  
- Random Forest  

### Boosting Models
- Gradient Boosting  

All models are trained using consistent data splits for fair comparison.


### Linear Models
- Logistic Regression 

### Tree-Based Models
- Decision Tree  
- Random Forest  

### Boosting Models
- Gradient Boosting  

## 6. Model Evaluation

Evaluate each model using selected metrics.

Analyze:

- Recall (churn detection ability)  
- Precision (cost control)  
- F1-score (balance)  
- ROC-AUC (overall performance)  


## 7. Model Comparison

Compare all models and approaches:

- Baseline vs advanced models  
- Effect of class imbalance handling  
- Trade-offs between recall and precision  

Select the best-performing model based on business priorities.

## 8. Hyperparameter Tuning

Optimize the selected model using:

- GridSearchCV or RandomizedSearchCV  

Focus on improving recall and overall model performance.

## 9. Final Model Evaluation

Evaluate the tuned model on the test set.

Report final metrics and compare with baseline.


## 10. Feature Importance & Interpretation

Analyze which features contribute most to predictions.

Identify key drivers of churn and validate earlier hypotheses.

## 11. Conclusion

Summarize:

- Best model and why  
- Final performance metrics  
- Impact of class imbalance handling  
- Key factors influencing churn  
- Potential business insights and actions  